In [ ]:
# ---------------------------------------------------------------------
# CELL 1: PASTE ALL OF THIS CODE INTO THE FIRST COLAB CELL AND RUN IT
# ---------------------------------------------------------------------

#@title A-to-Z Training Script for Neural Codec (TS3 GAN)
#@markdown ### 1. Install Dependencies
#@markdown This cell will install all required packages and define all models and training functions.
!pip install pesq[speechmetrics] pystoi librosa

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchaudio
import os
import numpy as np
import torch.nn.functional as F
import math
import traceback
import tarfile
import requests
import threading
import librosa # Added for metric calculation in validation
from pesq import pesq # Added for metric calculation in validation
from pystoi import stoi # Added for metric calculation in validation
from torch.optim.lr_scheduler import ExponentialLR
from google.colab import files # For downloading

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ---------------------------------------------------------------------
# CONFIGURATION OPTIONS
# ---------------------------------------------------------------------
#@markdown ---
#@markdown ### 2. Training Configuration
#@markdown Training uses the Adversarial Codec (GACodec) for high quality.
MODEL_ARCHITECTURE = "TS3_Codec (16kbps, Transformer GAN)" #@param ["TS3_Codec (16kbps, Transformer GAN)"]
DATASET_TO_DOWNLOAD = "train-clean-100" #@param ["dev-clean", "train-clean-100", "train-clean-360"]
#@markdown *Note: Use `train-clean-100` for best results. Training GANs requires larger data.*

#@markdown ---
#@markdown ### 3. Training Hyperparameters (CRITICAL TUNING FOR PESQ > 3.6)
EPOCHS = 1000 #@param {type:"integer"}
#@markdown *Set to 1000 for realistic high-quality training. Will be stopped by session limit.*
LEARNING_RATE = 0.0001 #@param {type:"number"}

#@markdown **Adversarial and Feature Loss Weights**
#@markdown *These are key for quality.*
LAMBDA_STFT = 45.0 #@param {type:"number"}
LAMBDA_FM = 4.0 #@param {type:"number"}
#@markdown *(Increased from 2.0 to 4.0 to force better perceptual feature matching)*

#@markdown **GAN Stability Controls**
DISC_WARMUP_STEPS = 100 #@param {type:"integer"}
LR_DECAY_RATE = 0.9999 #@param {type:"number"}
#@markdown **Discriminator to Generator Training Ratio**
D_G_RATIO = 2 #@param {type:"integer"}
#@markdown *(Train Discriminator 2x for every 1x Generator to maintain balance)*

#@markdown ---
#@markdown ### 4. Hardware & VRAM Settings
BATCH_SIZE = 8 #@param {type:"integer"}
#@markdown *Note: Increased model complexity may require even smaller batch sizes (4 or 6).*
GRAD_ACCUM_STEPS = 4 #@param {type:"integer"}
USE_AMP = True #@param {type:"boolean"}
NUM_WORKERS = 4 #@param {type:"integer"}

#@markdown **Latency Control (Crucial for <20ms end-to-end)**
TFM_HISTORY_CHUNKS = 0 #@param {type:"integer"}
#@markdown *SET TO 0: Guarantees 10ms frame latency (160 samples). Set to 1 (20ms total) or 2 (30ms total) if quality plateau is hit.*

#@markdown ---
#@markdown ### 5. File Paths
SAVE_PATH_PREFIX = "low_latency_codec" #@param {type:"string"}
DOWNLOAD_PATH = "/content/LibriSpeech" #@param {type:"string"}
#@markdown ---

# ---------------------------------------------------------------------
# DATASET DOWNLOAD FUNCTION (Modified to download validation set)
# ---------------------------------------------------------------------

def download_librispeech(dataset_name, path):
    """Downloads and extracts a LibriSpeech dataset."""
    url_map = {
        "dev-clean": "https://www.openslr.org/resources/12/dev-clean.tar.gz",
        "train-clean-100": "https://www.openslr.org/resources/12/train-clean-100.tar.gz",
        "train-clean-360": "https://www.openslr.org/resources/12/train-clean-360.tar.gz",
    }

    # We must ensure dev-clean is downloaded for validation, regardless of main choice
    datasets_to_check = [dataset_name]
    if dataset_name != "dev-clean":
        datasets_to_check.append("dev-clean")

    downloaded_paths = {}

    for name in datasets_to_check:
        if name not in url_map: continue

        url = url_map[name]
        filename = url.split("/")[-1]
        compressed_path = os.path.join(path, filename)
        target_folder_name = name
        expected_dataset_path = os.path.join(path, "LibriSpeech", target_folder_name)

        print(f"Checking for existing dataset folder for {name} at: {expected_dataset_path}")
        if os.path.exists(expected_dataset_path) and os.path.isdir(expected_dataset_path):
            print(f"Dataset {name} already found.")
            downloaded_paths[name] = expected_dataset_path
            continue

        try:
            if not os.path.exists(compressed_path):
                print(f"Downloading {filename}...")
                with requests.get(url, stream=True) as r:
                    r.raise_for_status()

                    # FIX: Ensure parent directory for compressed file exists
                    os.makedirs(os.path.dirname(compressed_path), exist_ok=True)

                    with open(compressed_path, 'wb') as f:
                        for chunk in r.iter_content(chunk_size=8192):
                            f.write(chunk)
                print("Download complete.")

            print(f"Extracting {name}...")
            with tarfile.open(compressed_path, "r:gz") as tar:
                tar.extractall(path=path)
            print("Extraction complete.")
            os.remove(compressed_path)

            if os.path.exists(expected_dataset_path) and os.path.isdir(expected_dataset_path):
                print(f"✓ Dataset {name} ready.")
                downloaded_paths[name] = expected_dataset_path
            else:
                print(f"ERROR: Could not find extracted folder for {name}.")

        except Exception as e:
            print(f"Error during download/extraction for {name}: {e}")
            traceback.print_exc()

    if dataset_name not in downloaded_paths:
        return None, None

    return downloaded_paths.get(dataset_name), downloaded_paths.get("dev-clean")


# ---------------------------------------------------------------------
# MODEL DEFINITIONS (OPTIMIZED FOR CPU SPEED)
# ---------------------------------------------------------------------

# --- Global Configuration (Unchanged) ---
HOP_SIZE = 160 # 10ms frame (160 samples / 16000 Hz = 0.01s) - CRITICAL LATENCY FIX
LATENT_DIM = 128 # High capacity

VQ_EMBEDDINGS = 256
NUM_VQ_STAGES = 2
VQ_INDICES_PER_STAGE = 10
NUM_QUANTIZERS = VQ_INDICES_PER_STAGE

# --- Loss Components (Unchanged) ---
class MultiResolutionSTFTLoss(nn.Module):
    def __init__(self, fft_sizes=[1024, 2048, 512], hop_sizes=[120, 240, 50], win_lengths=[600, 1200, 240]):
        super(MultiResolutionSTFTLoss, self).__init__()
        self.fft_sizes = fft_sizes
        self.hop_sizes = hop_sizes
        self.win_lengths = win_lengths
        self.window = torch.hann_window
        assert len(fft_sizes) == len(hop_sizes) == len(win_lengths)

    def forward(self, y_hat, y):
        sc_loss, mag_loss = 0.0, 0.0
        for fft, hop, win in zip(self.fft_sizes, self.hop_sizes, self.win_lengths):
            window = self.window(win, device=y.device)
            y_hat_float = y_hat.squeeze(1).to(torch.float32)
            y_float = y.squeeze(1).to(torch.float32)
            spec_hat = torch.stft(y_hat_float, n_fft=fft, hop_length=hop, win_length=win, window=window, return_complex=True)
            spec = torch.stft(y_float, n_fft=fft, hop_length=hop, win_length=win, window=window, return_complex=True)

            sc_loss += torch.norm(torch.abs(spec) - torch.abs(spec_hat), p='fro') / (torch.norm(torch.abs(spec), p='fro') + 1e-6)
            mag_loss += F.l1_loss(torch.log(torch.abs(spec).clamp(min=1e-9)), torch.log(torch.abs(spec_hat).clamp(min=1e-9)))

        return (sc_loss / len(self.fft_sizes)) + (mag_loss / len(self.fft_sizes))

# --- Causal Components (Unchanged) ---
class CausalConv1d(nn.Conv1d):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.causal_padding = self.kernel_size[0] - 1
    def forward(self, x):
        return super().forward(F.pad(x, (self.causal_padding, 0)))

class CausalConvTranspose1d(nn.ConvTranspose1d):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.causal_padding = self.kernel_size[0] - self.stride[0]
    def forward(self, x):
        x = super().forward(x)
        if self.causal_padding != 0:
            return x[..., :-self.causal_padding]
        return x

# --- Residual Vector Quantizer (RVQ) (Unchanged) ---
class ResidualVectorQuantizer(nn.Module):
    def __init__(self, num_stages, num_embeddings, embedding_dim, commitment_cost):
        super().__init__()
        self.num_stages = num_stages
        self.commitment_cost = commitment_cost
        self.vqs = nn.ModuleList([
            VectorQuantizerSingle(num_embeddings, embedding_dim, 0.0)
            for _ in range(num_stages)
        ])

    def forward(self, z_e):
        quantized_output = 0.0
        residual = z_e
        total_vq_loss = 0.0
        all_indices = []

        for vq in self.vqs:
            z_q_i, vq_loss_i, indices_i = vq(residual)
            residual = residual - z_q_i.detach()
            quantized_output = quantized_output + z_q_i
            total_vq_loss += vq_loss_i
            all_indices.append(indices_i)

        total_vq_loss = total_vq_loss * self.commitment_cost / self.num_stages
        return quantized_output, total_vq_loss, torch.stack(all_indices, dim=1)

class VectorQuantizerSingle(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, commitment_cost):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.commitment_cost = commitment_cost
        self.embedding = nn.Embedding(self.num_embeddings, self.embedding_dim)
        self.embedding.weight.data.uniform_(-1.0 / self.num_embeddings, 1.0 / self.num_embeddings)

    def forward(self, z_e):
        z_e_float = z_e.to(torch.float32)
        z_e_flat = z_e_float.permute(0, 2, 1).contiguous().view(-1, self.embedding_dim)
        distances = (torch.sum(z_e_flat**2, dim=1, keepdim=True)
                     + torch.sum(self.embedding.weight**2, dim=1)
                     - 2 * torch.matmul(z_e_flat, self.embedding.weight.t()))
        encoding_indices = torch.argmin(distances, dim=1)
        z_q = self.embedding(encoding_indices).view(z_e_float.shape[0], -1, self.embedding_dim)
        z_q = z_q.permute(0, 2, 1).contiguous()
        e_loss = F.mse_loss(z_q.detach(), z_e_float)
        z_q = z_e + (z_q - z_e_float).detach()
        return z_q, e_loss, encoding_indices.view(z_e.shape[0], -1)


# --- Encoder/Decoder (Unchanged - Architecturally Sound for 16kbps/10ms) ---
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        ResBlock = lambda c: nn.Sequential(CausalConv1d(c, c, 3), nn.ELU(), CausalConv1d(c, c, 1))
        # Total Downsampling: 16x. (160 samples -> 10 latent vectors)
        self.net = nn.Sequential(
            CausalConv1d(1, 64, 5), nn.ELU(), ResBlock(64),
            CausalConv1d(64, 128, 3, stride=2), nn.ELU(), ResBlock(128), # x2
            CausalConv1d(128, 256, 3, stride=2), nn.ELU(), ResBlock(256), # x4
            CausalConv1d(256, 512, 3, stride=2), nn.ELU(), ResBlock(512), # x8
            CausalConv1d(512, 512, 3, stride=2), nn.ELU(), ResBlock(512), # x16 total stride
            CausalConv1d(512, LATENT_DIM, 3, stride=1), nn.ELU(),
        )
    def forward(self, x): return self.net(x)

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        ResBlock = lambda c: nn.Sequential(CausalConv1d(c, c, 3), nn.ELU(), CausalConv1d(c, c, 1))
        # Total Upsampling: x16. (10 latent vectors -> 160 samples)
        self.net = nn.Sequential(
            CausalConvTranspose1d(LATENT_DIM, 512, 3, stride=1), nn.ELU(), ResBlock(512),
            CausalConvTranspose1d(512, 512, 3, stride=2), nn.ELU(), ResBlock(512), # 10 -> 20
            CausalConvTranspose1d(512, 256, 3, stride=2), nn.ELU(), ResBlock(256), # 20 -> 40
            CausalConvTranspose1d(256, 128, 3, stride=2), nn.ELU(), ResBlock(128), # 40 -> 80
            CausalConvTranspose1d(128, 64, 3, stride=2), nn.ELU(), ResBlock(64), # 80 -> 160 (x16 total)
            CausalConv1d(64, 1, 3), nn.Tanh()
        )
    def forward(self, x): return self.net(x)

# --- Causal Transformer (CRITICAL CPU LATENCY FIX) ---
class CausalTransformerEncoder(nn.Module):
    def __init__(self, d_model, nhead, num_layers=1, history_chunks=0):
        super().__init__()
        self.d_model = d_model
        # CRITICAL FIX: Reduced to 1 layer for max CPU performance
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True, activation=F.gelu)
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.state_len = history_chunks * VQ_INDICES_PER_STAGE
        print(f"CausalTransformerEncoder initialized with STATE_LEN: {self.state_len}, Layers: {num_layers}")


    def get_causal_mask(self, sz, device):
        return torch.triu(torch.ones(sz, sz, device=device), diagonal=1).to(torch.bool)

    def forward(self, x, state):
        if self.state_len > 0 and state is not None:
            state_history = state[:, -self.state_len:, :]
            inp = torch.cat([state_history, x], dim=1)
        else:
            inp = x

        new_state = inp.detach()
        mask = self.get_causal_mask(inp.shape[1], x.device)
        norm_inp = self.norm(inp)
        out = self.transformer(norm_inp, mask=mask, src_key_padding_mask=None)
        out = out[:, -x.shape[1]:, :]
        return out, new_state


# --- Generator: TS3 Codec (OPTIMIZED RVQ Version) ---
class TS3_Codec(nn.Module):
    """The Generator model now utilizing RVQ for increased capacity."""
    def __init__(self, history_chunks=0):
        super().__init__()
        self.encoder = Encoder()
        self.quantizer = ResidualVectorQuantizer(NUM_VQ_STAGES, VQ_EMBEDDINGS, LATENT_DIM, 0.25)
        # CRITICAL FIX: Reduced TFM layers from 2 to 1 for max CPU speed.
        self.encoder_tfm = CausalTransformerEncoder(LATENT_DIM, nhead=4, num_layers=1, history_chunks=history_chunks)
        self.decoder_tfm = CausalTransformerEncoder(LATENT_DIM, nhead=4, num_layers=1, history_chunks=history_chunks)
        self.decoder = Decoder()

    def forward(self, x, h_enc=None, h_dec=None):
        x = x.to(torch.float32)
        z_e = self.encoder(x)
        z_e_tfm_in = z_e.permute(0, 2, 1)
        z_e_tfm_out, h_enc_new = self.encoder_tfm(z_e_tfm_in, h_enc)
        z_e_tfm_out = z_e_tfm_out.permute(0, 2, 1)
        z_q, vq_loss, indices = self.quantizer(z_e_tfm_out)
        z_q_tfm_in = z_q.permute(0, 2, 1)
        z_q_tfm_out, h_dec_new = self.decoder_tfm(z_q_tfm_in, h_dec)
        z_q_tfm_out = z_q_tfm_out.permute(0, 2, 1)
        x_hat = self.decoder(z_q_tfm_out)
        return x_hat, vq_loss, (h_enc_new, h_dec_new)

    def encode(self, x, h_enc):
        """Streaming encode path."""
        x = x.to(torch.float32)
        z_e = self.encoder(x)
        z_e_tfm_in = z_e.permute(0, 2, 1)
        z_e_tfm_out, h_enc_new = self.encoder_tfm(z_e_tfm_in, h_enc)
        z_e_tfm_out = z_e_tfm_out.permute(0, 2, 1)
        with torch.no_grad():
            _, _, indices = self.quantizer(z_e_tfm_out)
        return indices.view(indices.shape[0], -1), h_enc_new

    def decode(self, indices, h_dec):
        """Streaming decode path - reconstructs VQ vectors from flattened indices."""
        indices = indices.view(indices.shape[0], NUM_VQ_STAGES, VQ_INDICES_PER_STAGE)
        z_q = 0.0
        for stage in range(NUM_VQ_STAGES):
            vq_layer = self.quantizer.vqs[stage]
            indices_i = indices[:, stage, :]
            z_q_i = vq_layer.embedding(indices_i)
            z_q = z_q + z_q_i.permute(0, 2, 1)

        z_q_tfm_in = z_q.permute(0, 2, 1)
        z_q_tfm_out, h_dec_new = self.decoder_tfm(z_q_tfm_in, h_dec)
        z_q_tfm_out = z_q_tfm_out.permute(0, 2, 1)
        x_hat = self.decoder(z_q_tfm_out)
        return x_hat, h_dec_new


# --- Discriminator: Multi-Scale Adversarial Network (Unchanged) ---
class Discriminator(nn.Module):
    """Single Discriminator operating on one scale/resolution."""
    def __init__(self, start_channels=16):
        super().__init__()
        self.net = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(1, start_channels, 15, stride=1, padding=7),
                nn.LeakyReLU(0.2),
            ),
            nn.Sequential(nn.Conv1d(start_channels, start_channels * 2, 41, stride=4, padding=20, groups=4), nn.LeakyReLU(0.2)),
            nn.Sequential(nn.Conv1d(start_channels * 2, start_channels * 4, 41, stride=4, padding=20, groups=16), nn.LeakyReLU(0.2)),
            nn.Sequential(nn.Conv1d(start_channels * 4, start_channels * 8, 41, stride=4, padding=20, groups=64), nn.LeakyReLU(0.2)),
            nn.Sequential(nn.Conv1d(start_channels * 8, start_channels * 8, 5, stride=1, padding=2), nn.LeakyReLU(0.2)),
        ])
        self.final_conv = nn.Conv1d(start_channels * 8, 1, 3, stride=1, padding=1)

    def forward(self, x):
        feature_maps = []
        for layer in self.net:
            x = layer(x)
            feature_maps.append(x)
        output = self.final_conv(x)
        feature_maps.append(output)
        return output, feature_maps

class MultiScaleDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.discriminators = nn.ModuleList([
            Discriminator(start_channels=16),
            Discriminator(start_channels=32),
            Discriminator(start_channels=64),
        ])
        self.downsample = nn.AvgPool1d(kernel_size=4, stride=2, padding=1, count_include_pad=False)

    def forward(self, x):
        outputs = []
        feature_maps = []

        out_d1, fm_d1 = self.discriminators[0](x)
        outputs.append(out_d1); feature_maps.append(fm_d1)

        x_2x = self.downsample(x)
        out_d2, fm_d2 = self.discriminators[1](x_2x)
        outputs.append(out_d2); feature_maps.append(fm_d2)

        x_4x = self.downsample(x_2x)
        out_d3, fm_d3 = self.discriminators[2](x_4x)
        outputs.append(out_d3); feature_maps.append(fm_d3)

        return outputs, feature_maps


# --- TRADITIONAL CODECS (Unchanged) ---
class MuLawCodec:
    def __init__(self, quantization_channels=256): self.mu = float(quantization_channels - 1)
    def encode(self, x):
        mu_t = torch.tensor(self.mu, device=x.device, dtype=torch.float32)
        encoded = torch.sign(x) * torch.log1p(mu_t * torch.abs(x)) / torch.log1p(mu_t)
        return (((encoded + 1) / 2 * self.mu) + 0.5).to(torch.uint8)
    def decode(self, z):
        z_float = z.to(torch.float32)
        mu_t = torch.tensor(self.mu, device=z.device, dtype=torch.float32)
        y = (z_float / self.mu) * 2.0 - 1.0
        return (torch.sign(y) * (1.0 / self.mu) * (torch.pow(1.0 + self.mu, torch.abs(y)) - 1.0)).unsqueeze(1)

class ALawCodec:
    def __init__(self): self.A = 87.6
    def encode(self, x):
        a_t = torch.tensor(self.A, device=x.device, dtype=torch.float32)
        abs_x = torch.abs(x)
        encoded = torch.zeros_like(x)
        cond = abs_x < (1 / self.A)
        encoded[cond] = torch.sign(x[cond]) * (a_t * abs_x[cond]) / (1 + torch.log(a_t))
        encoded[~cond] = torch.sign(x[~cond]) * (1 + torch.log(a_t * abs_x[~cond])) / (1 + torch.log(a_t))
        return (((encoded + 1) / 2 * 255) + 0.5).to(torch.uint8)
    def decode(self, z):
        z_float = z.to(torch.float32)
        a_t = torch.tensor(self.A, device=z.device, dtype=torch.float32)
        y = (z_float / 127.5) - 1.0
        abs_y = torch.abs(y)
        decoded = torch.zeros_like(y)
        cond = abs_y < (1 / (1 + torch.log(a_t)))
        decoded[cond] = torch.sign(y[cond]) * (abs_y[cond] * (1 + torch.log(a_t))) / a_t
        decoded[~cond] = torch.sign(y[~cond]) * torch.exp(abs_y[~cond] * (1 + torch.log(a_t)) - 1) / a_t
        return decoded.unsqueeze(1)

# --- DATASET & TRAINING LOGIC (Modified for Validation Data) ---
TRAIN_CHUNK_SIZE = 16000

class AudioChunkDataset(Dataset):
    def __init__(self, directory, chunk_size=TRAIN_CHUNK_SIZE, sample_rate=16000):
        self.chunk_size, self.sample_rate = chunk_size, sample_rate
        self.file_paths = [os.path.join(r, f) for r, _, fs in os.walk(directory) for f in fs if f.lower().endswith(('.wav', '.flac'))]
        if not self.file_paths: raise ValueError("No audio files found.")
    def __len__(self): return len(self.file_paths)
    def __getitem__(self, idx):
        try:
            waveform, sr = torchaudio.load(self.file_paths[idx])
            if sr != self.sample_rate: waveform = torchaudio.transforms.Resample(sr, self.sample_rate)(waveform)
            if waveform.shape[0] > 1: waveform = torch.mean(waveform, dim=0, keepdim=True)
            waveform = waveform.to(torch.float32)
            if waveform.shape[1] > self.chunk_size:
                start = np.random.randint(0, waveform.shape[1] - self.chunk_size)
                waveform = waveform[:, start:start + self.chunk_size]
            else:
                waveform = F.pad(waveform, (0, self.chunk_size - waveform.shape[1]))
            if waveform.shape[1] != self.chunk_size:
                waveform = F.pad(waveform, (0, self.chunk_size - waveform.shape[1]))
            return waveform
        except Exception as e:
            print(f"Warning: Skipping file {self.file_paths[idx]}. Error: {e}")
            return torch.zeros((1, self.chunk_size), dtype=torch.float32)


def generator_loss(disc_fake_features, disc_real_features_for_fm, fm_weight, gan_weight):
    """Calculates Generator Loss (Adversarial + Feature Matching)."""
    adv_loss = 0.0
    fm_loss = 0.0

    # Adversarial Loss (LSGAN/MSE Loss for Generator: Target real=1.0)
    for disc_fake in disc_fake_features:
        adv_loss += F.mse_loss(disc_fake[-1], torch.ones_like(disc_fake[-1]))

    # Feature Matching Loss (L1 distance between intermediate discriminator features)
    for (fake_features, real_features) in zip(disc_fake_features, disc_real_features_for_fm):
        for (fake_feature, real_feature) in zip(fake_features[:-1], real_features[:-1]):
            fm_loss += F.l1_loss(fake_feature, real_feature.detach())

    # Normalize losses
    num_disc = len(disc_fake_features)
    num_fm_layers = sum(len(f) - 1 for f in disc_fake_features)
    fm_loss = fm_loss / (num_fm_layers if num_fm_layers > 0 else 1.0)
    adv_loss = adv_loss / num_disc

    return adv_loss * gan_weight, fm_loss * fm_weight

def discriminator_loss(disc_real_features, disc_fake_features):
    """Calculates Discriminator Loss (Real vs Fake)."""
    loss = 0.0
    # LSGAN/MSE Loss for Discriminator: Target real=1.0, target fake=0.0
    for real_out, fake_out in zip(disc_real_features, disc_fake_features):
        real_loss = F.mse_loss(real_out[-1], torch.ones_like(real_out[-1]))
        fake_loss = F.mse_loss(fake_out[-1], torch.zeros_like(fake_out[-1]))
        loss += (real_loss + fake_loss)
    return loss / len(disc_real_features)

# --- NEW: Metric Calculation Utility (Uses Val DataLoader) ---
def calculate_metrics(model, dataloader, device, progress_callback, sr=16000):
    """Calculates PESQ and STOI on a subset of the DEV data."""
    model.eval()
    total_pesq, total_stoi, total_samples = 0, 0, 0
    max_test_batches = 10 # Limit test batches for speed

    test_data_iter = iter(dataloader)

    try:
        for batch_idx in range(max_test_batches):
            inputs = next(test_data_iter).to(device)

            with torch.no_grad():
                h_enc, h_dec = None, None
                x_hat, _, _ = model(inputs, h_enc, h_dec)

                x_hat = x_hat[..., :inputs.shape[-1]]

                original_np = inputs.squeeze().cpu().numpy()
                reconstructed_np = x_hat.squeeze().cpu().numpy()

                for i in range(original_np.shape[0]):
                    orig = original_np[i]
                    rec = reconstructed_np[i]

                    try:
                        pesq_score = pesq(sr, orig, rec, 'wb')
                        stoi_score = stoi(orig, rec, sr, extended=False)

                        total_pesq += pesq_score
                        total_stoi += stoi_score
                        total_samples += 1
                    except Exception as e:
                        progress_callback.emit(f"Warning: Metric calculation failed for one sample: {e}")
                        continue

        if total_samples > 0:
            avg_pesq = total_pesq / total_samples
            avg_stoi = total_stoi / total_samples
            progress_callback.emit(f"--- VALIDATION (DEV SET) --- PESQ: {avg_pesq:.3f} | STOI: {avg_stoi:.3f} ---")

        model.train()
    except StopIteration:
        progress_callback.emit("Warning: Reached end of validation dataloader during metric calculation.")
        model.train()


def train_model(train_dataset_path, val_dataset_path, epochs, learning_rate, batch_size, model_save_path, progress_callback, stop_event, model_type, use_amp=True, accum_steps=1, num_workers=4, disc_warmup_steps=100, lr_decay_rate=0.9999, tfm_history_chunks=0, lambda_stft=45.0, lambda_fm=4.0, d_g_ratio=2):
    """
    Main GAN Training function with stability controls and validation.
    """
    if model_type != 'transformer':
        progress_callback.emit("Internal Error: Training only supports 'transformer' model_type now."); return

    try:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        progress_callback.emit(f"Using device: {device}")

        # Training Data Setup
        train_dataset = AudioChunkDataset(directory=train_dataset_path)
        train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True, drop_last=True)
        progress_callback.emit(f"Train Dataset loaded with {len(train_dataset)} files.")

        # Validation Data Setup (Critical Fix)
        val_dataset = AudioChunkDataset(directory=val_dataset_path)
        # Use a large batch size for faster validation metrics
        val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=1, pin_memory=True, drop_last=True)
        progress_callback.emit(f"Validation Dataset (dev-clean) loaded with {len(val_dataset)} files.")


        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

        generator = TS3_Codec(history_chunks=tfm_history_chunks).to(device)
        discriminator = MultiScaleDiscriminator().to(device)

        optimizer_g = optim.Adam(generator.parameters(), lr=learning_rate, betas=(0.5, 0.9))
        optimizer_d = optim.Adam(discriminator.parameters(), lr=learning_rate, betas=(0.5, 0.9))

        scheduler_g = ExponentialLR(optimizer_g, gamma=lr_decay_rate)
        scheduler_d = ExponentialLR(optimizer_d, gamma=lr_decay_rate)

        stft_criterion = MultiResolutionSTFTLoss().to(device)
        l1_criterion = nn.L1Loss().to(device)

        progress_callback.emit(f"Starting GAN training for TS3_Codec (Generator) and Discriminator...")

        lambda_l1, lambda_vq = 1.0, 0.1
        gan_weight, fm_weight = 1.0, lambda_fm
        global_step = 0
        d_step_counter = 0

        # Initial validation check (Epoch 0)
        calculate_metrics(generator, val_dataloader, device, progress_callback)

        for epoch in range(epochs):
            if stop_event.is_set():
                progress_callback.emit("Training stopped by user."); break

            generator.train()
            discriminator.train()

            for i, data in enumerate(train_dataloader):
                if stop_event.is_set(): break
                inputs = data.to(device)
                global_step += 1
                d_step_counter += 1

                # --- 2. TRAIN DISCRIMINATOR (MSD) ---
                if d_step_counter % d_g_ratio == 0:
                    optimizer_d.zero_grad()

                    with torch.cuda.amp.autocast(enabled=use_amp):
                        h_enc, h_dec = None, None
                        x_hat, _, _ = generator(inputs, h_enc, h_dec)
                        input_len = inputs.shape[-1]
                        if x_hat.shape[-1] > input_len:
                            x_hat = x_hat[..., :input_len]
                        elif x_hat.shape[-1] < input_len:
                            x_hat = F.pad(x_hat, (0, input_len - x_hat.shape[-1]))

                        x_hat_detached = x_hat.detach()

                        _, disc_real_features = discriminator(inputs)
                        _, disc_fake_features = discriminator(x_hat_detached)
                        loss_d_total = discriminator_loss(disc_real_features, disc_fake_features)

                    scaler.scale(loss_d_total / accum_steps).backward()

                    if global_step % accum_steps == 0:
                        torch.nn.utils.clip_grad_norm_(discriminator.parameters(), max_norm=1.0)
                        scaler.step(optimizer_d)
                        scaler.update()
                        optimizer_d.zero_grad()
                        scheduler_d.step()

                # --- 1. TRAIN GENERATOR (TS3_Codec) ---
                if global_step > disc_warmup_steps:
                    optimizer_g.zero_grad()

                    with torch.cuda.amp.autocast(enabled=use_amp):
                        x_hat, vq_loss, _ = generator(inputs, h_enc, h_dec)

                        input_len = inputs.shape[-1]
                        if x_hat.shape[-1] > input_len:
                            x_hat = x_hat[..., :input_len]
                        elif x_hat.shape[-1] < input_len:
                            x_hat = F.pad(x_hat, (0, input_len - x_hat.shape[-1]))

                        stft_loss = stft_criterion(x_hat, inputs)
                        l1_loss = l1_criterion(x_hat, inputs)

                        disc_fake_outputs_g, disc_fake_features_g = discriminator(x_hat)
                        with torch.no_grad():
                            _, disc_real_features_for_fm = discriminator(inputs)

                        adv_loss, fm_loss = generator_loss(disc_fake_features_g, disc_real_features_for_fm, fm_weight=fm_weight, gan_weight=gan_weight)

                        loss_g_total = ((adv_loss + fm_loss) +
                                        lambda_stft * stft_loss +
                                        lambda_l1 * l1_loss +
                                        lambda_vq * vq_loss)

                    scaler.scale(loss_g_total / accum_steps).backward()

                    if global_step % accum_steps == 0:
                        torch.nn.utils.clip_grad_norm_(generator.parameters(), max_norm=1.0)
                        scaler.step(optimizer_g)
                        scaler.update()
                        optimizer_g.zero_grad()
                        scheduler_g.step()

                    if global_step % (20 * accum_steps * d_g_ratio) == 19:
                        progress_callback.emit(
                            f"[E {epoch + 1}, B {i + 1}] G Loss: {loss_g_total.item():.4f} (Adv: {adv_loss.item():.4f}, FM: {fm_loss.item():.4f}, STFT: {stft_loss.item():.4f}) | D Loss: {loss_d_total.item():.4f} | LR: {optimizer_g.param_groups[0]['lr']:.2e}"
                        )

            # --- CRITICAL VALIDATION STEP ---
            progress_callback.emit(f"--- Epoch {epoch + 1} finished ---")
            if (epoch + 1) % 10 == 0:
                 calculate_metrics(generator, val_dataloader, device, progress_callback)

            # --- CRITICAL CHECKPOINT SAVE ---
            progress_callback.emit(f"Saving epoch {epoch + 1} checkpoint to {model_save_path}...")
            torch.save(generator.state_dict(), model_save_path)
            progress_callback.emit(f"Checkpoint saved.")


        if not stop_event.is_set():
            progress_callback.emit("Training finished. Saving Generator (TS3_Codec)...")
            torch.save(generator.state_dict(), model_save_path)
            progress_callback.emit(f"Generator saved to {model_save_path}")
    except Exception as e:
        progress_callback.emit(f"ERROR in training: {e}")
        progress_callback.emit(traceback.format_exc())

    finally:
        if 'generator' in locals() and generator is not None:
             try:
                 progress_callback.emit(f"Attempting final save of Generator to {model_save_path}...")
                 torch.save(generator.state_dict(), model_save_path)
                 progress_callback.emit(f"Final Generator state saved successfully.")
             except Exception as final_save_e:
                 progress_callback.emit(f"!!! Error during final save: {final_save_e} !!!")


# ---------------------------------------------------------------------
# MAIN EXECUTION SCRIPT
# ---------------------------------------------------------------------
#@markdown ---
#@markdown ### 6. Run Training
#@markdown Click the "Run" button below (or Shift+Enter in this cell) to start the training process using the settings above.
print("Starting main training script...")

# 1. Map GUI string to internal model_type
model_type_internal = "transformer"

# 2. Update save path
final_save_path = f"{SAVE_PATH_PREFIX}_ts3_gacodec.pth"
print(f"Selected model: {MODEL_ARCHITECTURE}")
print(f"Model (Generator) will be saved to: {final_save_path}")

# 3. Download data (fetches main data and dev-clean)
print(f"Downloading/Verifying data in {DOWNLOAD_PATH}...")
train_dataset_path, val_dataset_path = download_librispeech(DATASET_TO_DOWNLOAD, DOWNLOAD_PATH)

if train_dataset_path and val_dataset_path:
    print(f"Train Dataset ready at: {train_dataset_path}")
    print(f"Validation Dataset ready at: {val_dataset_path}")

    # 4. Start training
    print("--- Starting Training ---")
    stop_event = threading.Event()

    class ColabProgressEmitter:
        def emit(self, msg):
            print(msg)

    colab_printer = ColabProgressEmitter()

    train_model(
        train_dataset_path=train_dataset_path,
        val_dataset_path=val_dataset_path, # Pass the separate validation path
        epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        batch_size=BATCH_SIZE,
        model_save_path=final_save_path,
        progress_callback=colab_printer,
        stop_event=stop_event,
        model_type=model_type_internal,
        use_amp=USE_AMP,
        accum_steps=GRAD_ACCUM_STEPS,
        num_workers=NUM_WORKERS,
        # Pass new stability params
        disc_warmup_steps=DISC_WARMUP_STEPS,
        lr_decay_rate=LR_DECAY_RATE,
        tfm_history_chunks=TFM_HISTORY_CHUNKS, # Pass latency control
        # Pass new quality params
        lambda_stft=LAMBDA_STFT,
        lambda_fm=LAMBDA_FM,
        d_g_ratio=D_G_RATIO
    )

    print("--- Training Finished ---")
    print(f"Model saved to {final_save_path}")
    print("\n=======================================================")
    print("To download your high-quality Generator model, run the next cell.")
    print("=======================================================")

else:
    print("Failed to download/find dataset. Training aborted.")


  Preparing metadata (setup.py) ... done
  Created wheel for pesq: filename=pesq-0.0.4-cp312-cp312-linux_x86_64.whl size=284116 sha256=b990ee436d22b0d840f97cd718e29c0ecfec9f159782878b6e7d71cc6a11a455
  Stored in directory: /root/.cache/pip/wheels/9b/d4/a4/9cf3512534cd47ce4a036d1593ee4013f2bf7509e631a147a3
Successfully built pesq
PyTorch Version: 2.8.0+cu126
CUDA Available: False
Starting main training script...
Selected model: TS3_Codec (16kbps, Transformer GAN)
Model (Generator) will be saved to: low_latency_codec_ts3_gacodec.pth
Downloading/Verifying data in /content/LibriSpeech...
Checking for existing dataset folder for train-clean-100 at: /content/LibriSpeech/LibriSpeech/train-clean-100
